# Spark vs Hadoop MapReduce — Banking Transaction Benchmark

**Datasets**: PaySim (~493 MB) + BankSim (~47 MB + ~31 MB)  
**Workloads**:
1. Daily summary aggregation  
2. Fraud-proxy detection (velocity + z-score)  
3. Large join + rolling 24h window  

**Objective**: Quantify Spark vs Hadoop MapReduce trade-offs for banking workloads

In [ ]:
import os
import sys
import time
import subprocess
import csv
import math
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *

BASE_DIR = os.environ.get('BENCHMARK_BASE_DIR', '/home/jovyan')
DATA_DIR = os.path.join(BASE_DIR, 'data')
LOGS_DIR = os.path.join(BASE_DIR, 'logs')
PLOTS_DIR = os.path.join(BASE_DIR, 'plots')
SRC_DIR = os.path.join(BASE_DIR, 'src')

sys.path.insert(0, SRC_DIR)

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f'BASE_DIR: {BASE_DIR}')
print(f'DATA_DIR: {DATA_DIR}')
print('Environment ready.')

## 1. Dataset Overview

In [ ]:
PAYSIM_PATH = os.path.join(DATA_DIR, 'paysim_transactions.csv')
BANKSIM_TX_PATH = os.path.join(DATA_DIR, 'banksim_transactions.csv')
BANKSIM_NET_PATH = os.path.join(DATA_DIR, 'banksim_network.csv')

dataset_info = []
for name, path in [('PaySim', PAYSIM_PATH), ('BankSim Tx', BANKSIM_TX_PATH), ('BankSim Net', BANKSIM_NET_PATH)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        with open(path) as f:
            row_count = sum(1 for _ in f) - 1
        dataset_info.append({'Name': name, 'Path': os.path.basename(path),
                             'Size (MB)': f'{size_mb:.1f}', 'Rows': f'{row_count:,}'})
    else:
        dataset_info.append({'Name': name, 'Path': os.path.basename(path),
                             'Size (MB)': 'NOT FOUND', 'Rows': '-'})

pd.DataFrame(dataset_info)

In [ ]:
if os.path.exists(PAYSIM_PATH):
    paysim_sample = pd.read_csv(PAYSIM_PATH, nrows=5000)
    print('PaySim columns:', paysim_sample.columns.tolist())
    print(f'PaySim fraud rate: {paysim_sample["isFraud"].mean():.3%}')
    print(f'PaySim amount stats:')
    print(paysim_sample['amount'].describe().round(2))
    display(paysim_sample.head(3))

In [ ]:
if os.path.exists(BANKSIM_TX_PATH):
    banksim_sample = pd.read_csv(BANKSIM_TX_PATH, nrows=5000,
                                  quotechar="'", skipinitialspace=True)
    print('BankSim Tx columns:', banksim_sample.columns.tolist())
    print(f'BankSim fraud rate: {banksim_sample["fraud"].astype(float).mean():.3%}')
    display(banksim_sample.head(3))

## 2. Spark Session Setup

In [ ]:
spark = (
    SparkSession.builder
    .appName('BankingBenchmark')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .config('spark.executor.memory', '2g')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'Spark UI: http://localhost:4040')

## 3. Benchmark Utilities

In [ ]:
import psutil
import threading
import tracemalloc
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Callable

@dataclass
class RunResult:
    engine: str
    workload: str
    sample_fraction: float
    data_size_mb: float
    wall_time_sec: float
    peak_memory_mb: float
    cpu_time_sec: float
    output_rows: int = 0
    exit_code: int = 0
    error_msg: str = ''

class ResourceMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self._stop = threading.Event()
        self._thread = None
        self.mem_samples = []
        self._proc = psutil.Process(os.getpid())

    def _loop(self):
        while not self._stop.is_set():
            try:
                self.mem_samples.append(self._proc.memory_info().rss / (1024*1024))
            except:
                break
            time.sleep(self.interval)

    def start(self):
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=3)

    @property
    def peak_mb(self):
        return max(self.mem_samples, default=0.0)

def timed_run(engine, workload, fn, data_paths, fraction=1.0) -> RunResult:
    if isinstance(data_paths, str):
        data_paths = [data_paths]
    data_size_mb = sum(
        os.path.getsize(p) / (1024*1024)
        for p in data_paths if os.path.exists(p)
    ) * fraction

    monitor = ResourceMonitor()
    monitor.start()
    t0_wall = time.perf_counter()
    t0_cpu  = time.process_time()

    rows = 0
    exit_code = 0
    error_msg = ''
    try:
        result = fn()
        if isinstance(result, int):
            rows = result
    except Exception as e:
        exit_code = 1
        error_msg = str(e)
        print(f'ERROR in {engine}/{workload}: {e}')

    wall_elapsed = time.perf_counter() - t0_wall
    cpu_elapsed  = time.process_time() - t0_cpu
    monitor.stop()

    return RunResult(
        engine=engine, workload=workload,
        sample_fraction=fraction, data_size_mb=data_size_mb,
        wall_time_sec=wall_elapsed, peak_memory_mb=monitor.peak_mb,
        cpu_time_sec=cpu_elapsed, output_rows=rows,
        exit_code=exit_code, error_msg=error_msg
    )

ALL_RESULTS: List[RunResult] = []
print('Benchmark utilities ready.')

## 4. Workload 1 — Daily Summary Aggregation

**Task**: Group by (account, day, type) → SUM(amount), COUNT(*), AVG(amount)

In [ ]:
PAYSIM_SCHEMA = StructType([
    StructField('step', LongType(), True),
    StructField('type', StringType(), True),
    StructField('amount', DoubleType(), True),
    StructField('nameOrig', StringType(), True),
    StructField('oldbalanceOrg', DoubleType(), True),
    StructField('newbalanceOrig', DoubleType(), True),
    StructField('nameDest', StringType(), True),
    StructField('oldbalanceDest', DoubleType(), True),
    StructField('newbalanceDest', DoubleType(), True),
    StructField('isFraud', IntegerType(), True),
    StructField('isFlaggedFraud', IntegerType(), True),
])

BANKSIM_SCHEMA = StructType([
    StructField('step', DoubleType(), True),
    StructField('customer', StringType(), True),
    StructField('age', StringType(), True),
    StructField('gender', StringType(), True),
    StructField('zipcodeOri', StringType(), True),
    StructField('merchant', StringType(), True),
    StructField('zipMerchant', StringType(), True),
    StructField('category', StringType(), True),
    StructField('amount', DoubleType(), True),
    StructField('fraud', IntegerType(), True),
])
print('Schemas defined.')

In [ ]:
def spark_wl1_aggregation(data_dir, fraction=1.0):
    paysim_path = os.path.join(data_dir, 'paysim_transactions.csv')
    banksim_path = os.path.join(data_dir, 'banksim_transactions.csv')
    results = []

    if os.path.exists(paysim_path):
        df = spark.read.csv(paysim_path, schema=PAYSIM_SCHEMA, header=True, mode='DROPMALFORMED')
        if fraction < 1.0:
            df = df.sample(fraction=fraction, seed=42)
        agg = (
            df.filter(F.col('step').isNotNull())
            .withColumn('day', (F.col('step') / 24).cast('long'))
            .groupBy('nameOrig', 'day', 'type')
            .agg(
                F.sum('amount').alias('total_amount'),
                F.count('*').alias('tx_count'),
                F.avg('amount').alias('avg_amount'),
                F.sum(F.when(F.col('isFraud') == 1, 1).otherwise(0)).alias('fraud_count')
            )
            .withColumn('dataset', F.lit('PAYSIM'))
        )
        results.append(agg.count())

    if os.path.exists(banksim_path):
        df = spark.read.csv(banksim_path, schema=BANKSIM_SCHEMA, header=True, mode='DROPMALFORMED')
        if fraction < 1.0:
            df = df.sample(fraction=fraction, seed=42)
        agg = (
            df.filter(F.col('step').isNotNull())
            .withColumn('day', F.col('step').cast('long'))
            .groupBy('customer', 'day', 'category')
            .agg(
                F.sum('amount').alias('total_amount'),
                F.count('*').alias('tx_count'),
                F.avg('amount').alias('avg_amount'),
                F.sum(F.when(F.col('fraud') == 1, 1).otherwise(0)).alias('fraud_count')
            )
            .withColumn('dataset', F.lit('BANKSIM'))
        )
        results.append(agg.count())

    return sum(results)

print('Spark WL1 function defined.')

In [ ]:
def hadoop_wl1_aggregation(data_dir, fraction=1.0):
    import random, tempfile, shutil
    random.seed(42)

    mapper = os.path.join(SRC_DIR, 'hadoop', 'workload1_aggregation', 'mapper.py')
    reducer = os.path.join(SRC_DIR, 'hadoop', 'workload1_aggregation', 'reducer.py')

    if not (os.path.exists(mapper) and os.path.exists(reducer)):
        raise FileNotFoundError(f'Hadoop scripts not found in {SRC_DIR}')

    tmp_dir = tempfile.mkdtemp(prefix='hadoop_wl1_')
    try:
        input_files = []
        for fname in ['paysim_transactions.csv', 'banksim_transactions.csv']:
            src = os.path.join(data_dir, fname)
            if not os.path.exists(src):
                continue
            dst = os.path.join(tmp_dir, fname)
            if fraction >= 1.0:
                shutil.copy(src, dst)
            else:
                with open(src) as fin, open(dst, 'w') as fout:
                    header = fin.readline()
                    fout.write(header)
                    for line in fin:
                        if random.random() < fraction:
                            fout.write(line)
            input_files.append(dst)

        output_lines = []
        for f in input_files:
            with open(f) as fin:
                map_proc = subprocess.run(
                    ['python3', mapper],
                    stdin=fin, capture_output=True, text=True
                )
            output_lines.append(map_proc.stdout)

        combined_map_output = '\n'.join(output_lines)
        sorted_map_output = '\n'.join(sorted(combined_map_output.strip().split('\n')))

        red_proc = subprocess.run(
            ['python3', reducer],
            input=sorted_map_output, capture_output=True, text=True
        )

        output_rows = len([l for l in red_proc.stdout.strip().split('\n') if l])
        return output_rows

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

print('Hadoop WL1 function defined.')

In [ ]:
for fraction in [0.1, 0.25, 0.5, 1.0]:
    print(f'\n--- WL1 @ {fraction:.0%} ---')

    r_spark = timed_run(
        engine='Spark_PySpark',
        workload='WL1_aggregation',
        fn=lambda f=fraction: spark_wl1_aggregation(DATA_DIR, f),
        data_paths=[PAYSIM_PATH, BANKSIM_TX_PATH],
        fraction=fraction
    )
    ALL_RESULTS.append(r_spark)
    print(f'  Spark:  {r_spark.wall_time_sec:.2f}s | {r_spark.peak_memory_mb:.0f} MB | rows={r_spark.output_rows}')

    r_hadoop = timed_run(
        engine='Hadoop_MapReduce',
        workload='WL1_aggregation',
        fn=lambda f=fraction: hadoop_wl1_aggregation(DATA_DIR, f),
        data_paths=[PAYSIM_PATH, BANKSIM_TX_PATH],
        fraction=fraction
    )
    ALL_RESULTS.append(r_hadoop)
    print(f'  Hadoop: {r_hadoop.wall_time_sec:.2f}s | {r_hadoop.peak_memory_mb:.0f} MB | rows={r_hadoop.output_rows}')

    if r_spark.wall_time_sec > 0 and r_hadoop.wall_time_sec > 0:
        print(f'  Speedup: {r_hadoop.wall_time_sec / r_spark.wall_time_sec:.1f}x (Spark faster)')

## 5. Workload 2 — Fraud Proxy Detection

**Rules**: velocity (tx count per account per hour ≥ 5) OR amount z-score > 2.5

In [ ]:
def spark_wl2_fraud(data_dir, fraction=1.0):
    VELOCITY_THRESHOLD = 5
    Z_THRESHOLD = 2.5
    results = []

    for fname, schema, acc_col, fraud_col in [
        ('paysim_transactions.csv', PAYSIM_SCHEMA, 'nameOrig', 'isFraud'),
        ('banksim_transactions.csv', BANKSIM_SCHEMA, 'customer', 'fraud'),
    ]:
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path):
            continue

        df = spark.read.csv(path, schema=schema, header=True, mode='DROPMALFORMED')
        if fraction < 1.0:
            df = df.sample(fraction=fraction, seed=42)

        df = df.filter(F.col('step').isNotNull() & F.col('amount').isNotNull())
        df = df.withColumn('hour_bucket', (F.col('step') % 24).cast('long'))
        df = df.withColumn('log_amount', F.log1p(F.col('amount')))

        vel_win = Window.partitionBy(acc_col, 'hour_bucket')
        df = df.withColumn('hourly_count', F.count('*').over(vel_win))
        df = df.withColumn('velocity_flag', F.col('hourly_count') >= VELOCITY_THRESHOLD)

        stats = df.agg(F.mean('log_amount').alias('m'), F.stddev('log_amount').alias('s')).collect()[0]
        gm, gs = stats['m'] or 0.0, stats['s'] or 1.0

        df = df.withColumn('z_score', (F.col('log_amount') - gm) / gs)
        df = df.withColumn('amount_flag', F.col('z_score') > Z_THRESHOLD)
        df = df.withColumn('fraud_proxy',
                           F.when(F.col('velocity_flag') | F.col('amount_flag'), 1).otherwise(0))

        results.append(df.count())

    return sum(results)

def hadoop_wl2_fraud(data_dir, fraction=1.0):
    import random, tempfile, shutil
    random.seed(42)

    mapper = os.path.join(SRC_DIR, 'hadoop', 'workload2_fraud', 'mapper.py')
    reducer = os.path.join(SRC_DIR, 'hadoop', 'workload2_fraud', 'reducer.py')

    if not (os.path.exists(mapper) and os.path.exists(reducer)):
        raise FileNotFoundError(f'Hadoop scripts not found')

    tmp_dir = tempfile.mkdtemp(prefix='hadoop_wl2_')
    try:
        input_files = []
        for fname in ['paysim_transactions.csv', 'banksim_transactions.csv']:
            src = os.path.join(data_dir, fname)
            if not os.path.exists(src):
                continue
            dst = os.path.join(tmp_dir, fname)
            if fraction >= 1.0:
                shutil.copy(src, dst)
            else:
                with open(src) as fin, open(dst, 'w') as fout:
                    fout.write(fin.readline())
                    for line in fin:
                        if random.random() < fraction:
                            fout.write(line)
            input_files.append(dst)

        all_map_lines = []
        for f in input_files:
            with open(f) as fin:
                proc = subprocess.run(['python3', mapper], stdin=fin, capture_output=True, text=True)
                all_map_lines.extend(proc.stdout.strip().split('\n'))

        sorted_lines = '\n'.join(sorted(l for l in all_map_lines if l))
        red_proc = subprocess.run(['python3', reducer], input=sorted_lines,
                                  capture_output=True, text=True)
        return len([l for l in red_proc.stdout.strip().split('\n') if l])
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

print('WL2 functions defined.')

In [ ]:
for fraction in [0.1, 0.25, 0.5, 1.0]:
    print(f'\n--- WL2 @ {fraction:.0%} ---')

    r_spark = timed_run('Spark_PySpark', 'WL2_fraud',
                        lambda f=fraction: spark_wl2_fraud(DATA_DIR, f),
                        [PAYSIM_PATH, BANKSIM_TX_PATH], fraction)
    ALL_RESULTS.append(r_spark)
    print(f'  Spark:  {r_spark.wall_time_sec:.2f}s | {r_spark.peak_memory_mb:.0f} MB')

    r_hadoop = timed_run('Hadoop_MapReduce', 'WL2_fraud',
                         lambda f=fraction: hadoop_wl2_fraud(DATA_DIR, f),
                         [PAYSIM_PATH, BANKSIM_TX_PATH], fraction)
    ALL_RESULTS.append(r_hadoop)
    print(f'  Hadoop: {r_hadoop.wall_time_sec:.2f}s | {r_hadoop.peak_memory_mb:.0f} MB')

## 6. Workload 3 — Large Join + Rolling 24h Window

In [ ]:
BANKSIM_NET_SCHEMA = StructType([
    StructField('Source', StringType(), True),
    StructField('Target', StringType(), True),
    StructField('Weight', DoubleType(), True),
    StructField('typeTrans', StringType(), True),
    StructField('fraud', IntegerType(), True),
])

def spark_wl3_join_window(data_dir, fraction=1.0):
    WINDOW_STEPS = 24
    results = []

    paysim_path = os.path.join(data_dir, 'paysim_transactions.csv')
    if os.path.exists(paysim_path):
        df = spark.read.csv(paysim_path, schema=PAYSIM_SCHEMA, header=True, mode='DROPMALFORMED')
        if fraction < 1.0:
            df = df.sample(fraction=fraction, seed=42)
        df = df.filter(F.col('step').isNotNull() & F.col('amount').isNotNull())

        dest_dim = (
            df.groupBy('nameDest')
            .agg(
                F.count('*').alias('dest_tx_count'),
                F.sum('amount').alias('dest_total_amount'),
                F.sum(F.when(F.col('isFraud') == 1, 1).otherwise(0)).alias('dest_fraud_count')
            )
        )

        joined = df.join(dest_dim, df['nameDest'] == dest_dim['nameDest'], how='left').drop(dest_dim['nameDest'])

        w = Window.partitionBy('nameOrig').orderBy('step').rangeBetween(-WINDOW_STEPS, 0)
        joined = joined.withColumn('rolling_24h_sum', F.sum('amount').over(w))
        joined = joined.withColumn('rolling_24h_count', F.count('*').over(w))

        results.append(joined.count())

    banksim_tx_path = os.path.join(data_dir, 'banksim_transactions.csv')
    banksim_net_path = os.path.join(data_dir, 'banksim_network.csv')
    if os.path.exists(banksim_tx_path) and os.path.exists(banksim_net_path):
        tx = spark.read.csv(banksim_tx_path, schema=BANKSIM_SCHEMA, header=True, mode='DROPMALFORMED')
        net = spark.read.csv(banksim_net_path, schema=BANKSIM_NET_SCHEMA, header=True, mode='DROPMALFORMED')
        if fraction < 1.0:
            tx = tx.sample(fraction=fraction, seed=42)

        merch_dim = (
            net.groupBy('Source')
            .agg(F.count('*').alias('net_tx_count'), F.sum('Weight').alias('net_weight_sum'))
        )

        joined = tx.join(merch_dim, tx['merchant'] == merch_dim['Source'], how='left').drop('Source')

        w = Window.partitionBy('customer').orderBy(F.col('step').cast('long')).rangeBetween(-WINDOW_STEPS, 0)
        joined = joined.withColumn('rolling_24h_sum', F.sum('amount').over(w))
        joined = joined.withColumn('rolling_24h_count', F.count('*').over(w))

        results.append(joined.count())

    return sum(results)

def hadoop_wl3_join_window(data_dir, fraction=1.0):
    import random, tempfile, shutil
    random.seed(42)

    mapper = os.path.join(SRC_DIR, 'hadoop', 'workload3_join_window', 'mapper.py')
    reducer = os.path.join(SRC_DIR, 'hadoop', 'workload3_join_window', 'reducer.py')

    if not (os.path.exists(mapper) and os.path.exists(reducer)):
        raise FileNotFoundError('Hadoop WL3 scripts not found')

    tmp_dir = tempfile.mkdtemp(prefix='hadoop_wl3_')
    try:
        input_files = []
        for fname in ['paysim_transactions.csv', 'banksim_transactions.csv', 'banksim_network.csv']:
            src = os.path.join(data_dir, fname)
            if not os.path.exists(src):
                continue
            dst = os.path.join(tmp_dir, fname)
            if fraction >= 1.0 or fname == 'banksim_network.csv':
                shutil.copy(src, dst)
            else:
                with open(src) as fin, open(dst, 'w') as fout:
                    fout.write(fin.readline())
                    for line in fin:
                        if random.random() < fraction:
                            fout.write(line)
            input_files.append(dst)

        all_map_lines = []
        for f in input_files:
            with open(f) as fin:
                proc = subprocess.run(['python3', mapper], stdin=fin, capture_output=True, text=True)
                all_map_lines.extend(proc.stdout.strip().split('\n'))

        sorted_lines = '\n'.join(sorted(l for l in all_map_lines if l))
        red_proc = subprocess.run(['python3', reducer], input=sorted_lines,
                                  capture_output=True, text=True)
        return len([l for l in red_proc.stdout.strip().split('\n') if l])
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

print('WL3 functions defined.')

In [ ]:
for fraction in [0.1, 0.25, 0.5, 1.0]:
    print(f'\n--- WL3 @ {fraction:.0%} ---')

    r_spark = timed_run('Spark_PySpark', 'WL3_join_window',
                        lambda f=fraction: spark_wl3_join_window(DATA_DIR, f),
                        [PAYSIM_PATH, BANKSIM_TX_PATH, BANKSIM_NET_PATH], fraction)
    ALL_RESULTS.append(r_spark)
    print(f'  Spark:  {r_spark.wall_time_sec:.2f}s | {r_spark.peak_memory_mb:.0f} MB')

    r_hadoop = timed_run('Hadoop_MapReduce', 'WL3_join_window',
                         lambda f=fraction: hadoop_wl3_join_window(DATA_DIR, f),
                         [PAYSIM_PATH, BANKSIM_TX_PATH, BANKSIM_NET_PATH], fraction)
    ALL_RESULTS.append(r_hadoop)
    print(f'  Hadoop: {r_hadoop.wall_time_sec:.2f}s | {r_hadoop.peak_memory_mb:.0f} MB')

## 7. Save Results

In [ ]:
results_csv = os.path.join(LOGS_DIR, 'benchmark_results.csv')

with open(results_csv, 'w', newline='') as f:
    if ALL_RESULTS:
        fieldnames = list(asdict(ALL_RESULTS[0]).keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in ALL_RESULTS:
            writer.writerow(asdict(r))

print(f'Saved {len(ALL_RESULTS)} results to {results_csv}')

results_df = pd.DataFrame([asdict(r) for r in ALL_RESULTS])
display(results_df[['engine', 'workload', 'sample_fraction', 'wall_time_sec',
                     'peak_memory_mb', 'cpu_time_sec', 'output_rows', 'exit_code']])

## 8. Visualizations

In [ ]:
import sys
viz_script = os.path.join(BASE_DIR, 'scripts', 'visualize.py')
if os.path.exists(viz_script):
    result = subprocess.run(
        [sys.executable, viz_script, '--input', results_csv, '--output-dir', PLOTS_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
else:
    print(f'Visualize script not found at {viz_script}')

In [ ]:
from IPython.display import Image, display as ipy_display
import glob

plot_files = sorted(glob.glob(os.path.join(PLOTS_DIR, '*.png')))
for pf in plot_files:
    print(f'Plot: {os.path.basename(pf)}')
    ipy_display(Image(filename=pf, width=900))

## 9. Summary Statistics & Speedup Analysis

In [ ]:
if len(results_df) > 0:
    full_df = results_df[results_df['sample_fraction'] >= 0.99]
    if len(full_df) == 0:
        full_df = results_df[results_df['sample_fraction'] == results_df['sample_fraction'].max()]

    spark_full = full_df[full_df['engine'] == 'Spark_PySpark']
    hadoop_full = full_df[full_df['engine'] == 'Hadoop_MapReduce']

    print('=== FULL DATASET COMPARISON ===')
    for wl in sorted(full_df['workload'].unique()):
        s = spark_full[spark_full['workload'] == wl]
        h = hadoop_full[hadoop_full['workload'] == wl]
        if len(s) and len(h):
            st = s.iloc[0]['wall_time_sec']
            ht = h.iloc[0]['wall_time_sec']
            speedup = ht / st if st > 0 else float('inf')
            print(f'{wl}:')
            print(f'  Spark:  {st:.1f}s | {s.iloc[0]["peak_memory_mb"]:.0f} MB')
            print(f'  Hadoop: {ht:.1f}s | {h.iloc[0]["peak_memory_mb"]:.0f} MB')
            print(f'  Speedup: {speedup:.1f}x (Spark faster)\n')

In [ ]:
spark.stop()
print('Spark session stopped. Benchmark complete.')